# 02 - Modeling

**Purpose:** Build the train/test split, the preprocessing pipeline, and the
first models (Logistic Regression baseline, Decision Tree, Random Forest).

**Sections:**
1. Load + rebuild EDA temporal features
2. Column decisions (drop PII, junk, high-cardinality)
3. Stratified 80/20 train/test split
4. Preprocessing pipeline (lives in `src/preprocessing.py`)
5. Baseline model + evaluation

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_DIR = Path("..") / "data" / "raw"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Load and combine (same as 01_eda.ipynb)
train = pd.read_csv(DATA_DIR / "fraudTrain.csv")
test = pd.read_csv(DATA_DIR / "fraudTest.csv")
df = pd.concat([train, test], ignore_index=True)
del train, test

# Rebuild the EDA temporal features so this notebook is self-contained
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
df["month"] = df["trans_date_trans_time"].dt.month

print("Loaded:", df.shape)
print("Fraud rate:", round(df["is_fraud"].mean(), 5))

Loaded: (1852394, 26)
Fraud rate: 0.00521


## 2. Column decisions

Dropping:

- **Junk:** `Unnamed: 0` (CSV row index)
- **PII:** `cc_num`, `first`, `last`, `street`, `city`, `zip`, `dob`, `trans_num`
- **High-cardinality:** `merchant` (693), `job` (497)
- **Redundant:** `unix_time` (already have `hour`/`day_of_week`/`month`), `trans_date_trans_time` (already extracted), `lat`/`long`/`merch_lat`/`merch_long` (near-duplicates of each other, no class separation)

Keeping:

- **Numeric:** `amt`, `city_pop`, `hour`, `day_of_week`, `month`
- **Categorical:** `gender`, `category`, `state`
- **Target:** `is_fraud`

In [2]:
DROP_COLS = [
    # junk
    "Unnamed: 0",
    # PII
    "cc_num", "first", "last", "street", "city", "zip", "dob", "trans_num",
    # high-cardinality
    "merchant", "job",
    # redundant / unhelpful
    "unix_time", "trans_date_trans_time",
    "lat", "long", "merch_lat", "merch_long",
]

NUMERIC_COLS = ["amt", "city_pop", "hour", "day_of_week", "month"]
CATEGORICAL_COLS = ["gender", "category", "state"]
TARGET = "is_fraud"

# Sanity check: nothing important is being dropped by accident
assert TARGET not in DROP_COLS
assert all(c in df.columns for c in DROP_COLS)
assert all(c in df.columns for c in NUMERIC_COLS + CATEGORICAL_COLS)

df_model = df.drop(columns=DROP_COLS).copy()

print("Kept columns:", list(df_model.columns))
print("Shape after drop:", df_model.shape)
print()
print(df_model.head(3))

Kept columns: ['category', 'amt', 'gender', 'state', 'city_pop', 'is_fraud', 'hour', 'day_of_week', 'month']
Shape after drop: (1852394, 9)

        category     amt gender state  city_pop  is_fraud  hour  day_of_week  month
0       misc_net    4.97      F    NC      3495         0     0            1      1
1    grocery_pos  107.23      F    WA       149         0     0            1      1
2  entertainment  220.11      M    ID      4154         0     0            1      1


## 3. Train/test split

Stratified 80/20 split on `is_fraud` so the ~0.52% fraud rate is preserved in both halves.

`random_state=42` for reproducibility. The test set is locked and never touched again until the final evaluation.

In [3]:
X = df_model.drop(columns=TARGET)
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print()
print("Train fraud rate:", round(y_train.mean(), 5), f"({y_train.sum():,} fraud)")
print("Test  fraud rate:", round(y_test.mean(), 5),  f"({y_test.sum():,} fraud)")
print()
print("Split preserved the imbalance ratio:", np.isclose(y_train.mean(), y_test.mean(), atol=1e-5))

Train: (1481915, 8)  Test: (370479, 8)

Train fraud rate: 0.00521 (7,721 fraud)
Test  fraud rate: 0.00521 (1,930 fraud)

Split preserved the imbalance ratio: True


### Train/test split - observation

Output looks exactly like it should:

- **Train:** 1,481,915 rows (80%) x 8 features
- **Test:** 370,479 rows (20%) x 8 features
- Fraud counts: 7,721 in train, 1,930 in test. Added together that's 9,651 - which is the exact total fraud count from the full dataset. Nothing lost, nothing double counted.
- **Both splits show a fraud rate of 0.00521** - identical to 5 decimal places. So `stratify=y` did its job. Without it, a random 20% slice could have ended up with meaningfully more or fewer fraud cases, which would quietly skew every downstream comparison.

The one thing I want to remember going forward: **`X_test` and `y_test` are now frozen.** I won't touch them until Step 12 (final evaluation). Every experiment from here on uses `X_train`/`y_train`, and for validation during tuning I'll carve out a slice of the train set inside cross-validation rather than peeking at test. That's the whole point of a held-out set - if I look at it to make decisions, it stops being held out and the final numbers become optimistic.